In [ ]:
# 1. INSTAL·LACIÓ
!pip install -q -U google-genai flask-cors pyngrok beautifulsoup4 requests
!pip install requests==2.32.4

import json, requests, time, re
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata
from google import genai

# --- CONFIGURACIÓ DE SEGURETAT ---
try:
    client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
    ngrok.set_auth_token(userdata.get('token_ngrok'))
    print("✅ API i ngrok connectats.")
except Exception as e:
    print(f"❌ ERROR SECRETS: {e}")

app = Flask(__name__)
CORS(app)

# --- EXTRACTOR TOTAL (Amb els noms de les pàgines en directe) ---
URL_BASE = "https://pjimenez.inscastellbisbal.net/"
dades_pau = []

def executar_extractor_total():
    global dades_pau
    urls_visitades = set()
    urls_per_visitar = [URL_BASE]
    domini = urlparse(URL_BASE).netloc

    print(f"🚀 Iniciant extracció massiva de {URL_BASE}...")

    # Límit pujat a 200 per agafar totes les teves entrades
    while urls_per_visitar and len(urls_visitades) < 200:
        url = urls_per_visitar.pop(0)

        if url in urls_visitades or any(x in url.lower() for x in ['.jpg', '.png', '.pdf', 'wp-admin', 'wp-json']):
            continue

        try:
            res = requests.get(url, timeout=10)
            if res.status_code != 200: continue

            soup = BeautifulSoup(res.text, 'html.parser')
            urls_visitades.add(url)

            titol = soup.title.string.strip() if soup.title else "Pàgina sense títol"
            blocs_text = [t.get_text(strip=True) for t in soup.find_all(['p', 'h1', 'h2', 'h3', 'li'])]
            contingut_net = " ".join([t for t in blocs_text if len(t) > 10])

            if len(contingut_net) > 50:
                # Guardem la info
                dades_pau.append({"url": url, "titol": titol, "contingut": contingut_net})

                # AQUESTA ÉS LA LÍNIA QUE VOLIES: Mostra el títol exacte de cada pàgina extreta
                print(f"✅ [{len(urls_visitades)}] Guardat: {titol[:60]}")

            # Buscar més enllaços
            for a in soup.find_all('a', href=True):
                enllac = urljoin(URL_BASE, a['href'])
                if urlparse(enllac).netloc == domini and enllac not in urls_visitades:
                    urls_per_visitar.append(enllac)

        except Exception:
            continue

    with open('dades_pau_total.json', 'w', encoding='utf-8') as f:
        json.dump(dades_pau, f, ensure_ascii=False, indent=4)
    print(f"\n📁 BBDD FINALITZADA: {len(dades_pau)} pàgines i entrades guardades amb èxit!")

# --- CERCADOR INTEL·LIGENT ---
def trobar_pagines_rellevants(pregunta, maxim=3):
    paraules = re.findall(r'\w+', pregunta.lower())
    resultats = []

    for pagina in dades_pau:
        text_pagina = (pagina['titol'] + " " + pagina['contingut']).lower()
        puntuacio = sum(1 for p in paraules if len(p) > 3 and p in text_pagina)
        resultats.append((puntuacio, pagina))

    resultats.sort(key=lambda x: x[0], reverse=True)
    return [r[1] for r in resultats[:maxim]]

# --- LÒGICA IA ---
def demanar_a_ia(pregunta):
    pagines_filtrades = trobar_pagines_rellevants(pregunta, maxim=3)

    context = "Ets l'assistent del portafolis d'en Pau Jiménez. Respon de forma amable en català basant-te NOMÉS en aquesta informació real del seu web:\n\n"
    for d in pagines_filtrades:
        context += f"Títol: {d['titol']}\nInfo: {d['contingut'][:600]}...\n---\n"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"{context}\n\nPregunta: {pregunta}"
    )
    return response.text

# --- RUTES I SERVIDOR ---
@app.route('/ask', methods=['POST'])
def ask():
    try:
        msg = request.json.get("message")
        print(f"📩 Usuari pregunta: {msg}")
        resposta = demanar_a_ia(msg)
        return jsonify({"reply": resposta})
    except Exception as e:
        print(f"❌ Error BackEnd: {e}")
        return jsonify({"reply": f"Error: {str(e)}"}), 500

if __name__ == '__main__':
    # 1. Fem l'extracció de tota la web
    executar_extractor_total()

    # 2. Encenem el servidor
    !pkill ngrok
    time.sleep(2)
    public_url = ngrok.connect(5000).public_url
    print(f"\n🌍 URL PER AL TEU JAVASCRIPT:\n{public_url}/ask\n")
    app.run(port=5000)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
✅ API i ngrok connectats.
🚀 Iniciant extracció massiva de https://pjimenez.inscastellbisbal.net/...
✅ [1] Guardat: El meu lloc web - Pau Jiménez Presas - SMX 1r
✅ [2] Guardat: El meu lloc web - Pau Jiménez Presas - SMX 1r
✅ [3] Guardat: iPOP! - Pau Jiménez Presas - SMX 1r
✅ [4] Guardat: Apunts - Pau Jiménez Presas - SMX 1r
✅ [5] Guardat: Reptes 1r SMX - Pau Jiménez Presas - SMX 1r
✅ [6] Guardat: REPTE 1.1 - Pau Jiménez Presas - SMX 1r
✅ [7]

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 07:52:40] "OPTIONS /ask HTTP/1.1" 200 -


📩 Usuari pregunta: hola


INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 07:52:43] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 07:53:06] "OPTIONS /ask HTTP/1.1" 200 -


📩 Usuari pregunta: hay corriculum


INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 07:53:10] "POST /ask HTTP/1.1" 200 -
